In [5]:
import os, glob, shutil

# Clone the shaurya branch and work from the repo root
REPO = "/content/IEEE-EOH-Acoustic-Array-Lung-Monitor"
if not os.path.exists(REPO):
    !git clone -b shaurya https://github.com/sbiruduganti/IEEE-EOH-Acoustic-Array-Lung-Monitor.git {REPO}
%cd {REPO}
!pip install -q brevitas onnx onnxscript kaggle

# Download only if the zip isn't already here
if not os.path.exists("respiratory-sound-database.zip"):
    !kaggle datasets download -d vbookshelf/respiratory-sound-database

# Extract (makedirs creates data/ and data/_raw)
os.makedirs("data/_raw", exist_ok=True)
!unzip -q -o respiratory-sound-database.zip -d data/_raw

# Flatten wav/txt into data/icbhi_dataset and fix metadata names
D = "data/icbhi_dataset"
os.makedirs(D, exist_ok=True)
for p in glob.glob("data/_raw/**/*", recursive=True):
    if os.path.isfile(p) and p.endswith((".wav", ".txt")):
        shutil.copy(p, D)
shutil.copy(glob.glob("data/_raw/**/patient_diagnosis.csv", recursive=True)[0],
            f"{D}/patient_diagnosis.csv")
shutil.copy(f"{D}/demographic_info.txt", f"{D}/ICBHI_challenge_demographics.txt")
shutil.rmtree("data/_raw")

# Verify
print("wavs:", len(glob.glob(f"{D}/*.wav")), "(expect 920)")
print(sorted(f for f in os.listdir(D) if not f.endswith(".wav"))[:6], "...")

Cloning into '/content/IEEE-EOH-Acoustic-Array-Lung-Monitor'...
remote: Enumerating objects: 362, done.
remote: Total 362 (delta 0), reused 0 (delta 0), pack-reused 362 (from 1)
Receiving objects: 100% (362/362), 9.62 MiB | 14.34 MiB/s, done.
Resolving deltas: 100% (211/211), done.
/content/IEEE-EOH-Acoustic-Array-Lung-Monitor
Dataset URL: https://www.kaggle.com/datasets/vbookshelf/respiratory-sound-database
License(s): unknown
100% 3.69G/3.69G [00:31<00:00, 127MB/s]

wavs: 920 (expect 920)
['101_1b1_Al_sc_Meditron.txt', '101_1b1_Pr_sc_Meditron.txt', '102_1b1_Ar_sc_Meditron.txt', '103_2b2_Ar_mc_LittC2SE.txt', '104_1b1_Al_sc_Litt3200.txt', '104_1b1_Ar_sc_Litt3200.txt'] ...


In [6]:
# --- Cell 2: sanity check before training ---
import numpy as np
from src.dataset import ICBHIDataset
ds = ICBHIDataset(D, gate_mode="cascade_eval")   # builds the cache on first run
print(len(ds), np.bincount([s["label"] for s in ds.samples]))
print(ds[0]["spectrogram"].shape, ds[0]["demographics"])  # (1,128,128), nonzero values

[ICBHIDataset] Building 4kHz (1, 128, 128) circpad STFT cache across 924 files...
[ICBHIDataset] Saved 6596 circularly-padded cycles to data/icbhi_dataset/icbhi_4khz_128x128_circpad_cache.pt.
6596 [ 322 5746  285  243]
torch.Size([1, 128, 128]) tensor([0.0300, 0.0000, 0.0000, 1.0000, 0.0000])


In [7]:
!sed -i 's/, bias_quant=Int8Bias//g' src/model_qat.py
!grep -c "bias_quant" src/model_qat.py   # should print 0

0


In [9]:
# --- Cell 3: train the 3-gate QAT cascade ---
!python -m src.train_qat --data_dir {D} --epochs 25 --bits 4

/usr/local/lib/python3.13/dist-packages/brevitas/graph/equalize.py:70: UserWarning: fast_hadamard_transform package not found, using standard pytorch kernels
  warnings.warn("fast_hadamard_transform package not found, using standard pytorch kernels")
[QAT Training] Device: cuda | Quantization: W4A4 | Epochs: 25
[ICBHIDataset] Loading precomputed 4kHz circpad STFT cache from data/icbhi_dataset/icbhi_4khz_128x128_circpad_cache.pt...

=== Gate 1: Healthy (0) vs. Diseased (1) | Train: 5276 | Val: 1320 ===
Traceback (most recent call last):
  File "<frozen runpy>", line 203, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/content/IEEE-EOH-Acoustic-Array-Lung-Monitor/src/train_qat.py", line 467, in <module>
    run_qat_finetuning(
    ~~~~~~~~~~~~~~~~~~^
        data_dir=args.data_dir,
        ^^^^^^^^^^^^^^^^^^^^^^^
    ...<4 lines>...
        bit_width=args.bits,
        ^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/content/IEEE-EOH-Acoustic-Array-Lung-Monitor/sr